# K-Nearest Neighbors (KNN) Classification Lab

This lab demonstrates practical end-to-end implementation of **K-Nearest Neighbors (KNN)** using Scikit-Learn. We will explore:
1. The critical role of **Feature Scaling** in distance-based algorithms.
2. Building leak-free workflows using Scikit-Learn's `Pipeline`.
3. Hyperparameter optimization with `GridSearchCV`.
4. Visualizing the **Error Rate vs. K Value** curve to select the optimal neighborhood size.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set plot styles
sns.set_theme(style="whitegrid")
%matplotlib inline

## 2. Load and Inspect the Dataset

We use the classic Iris dataset, comprising 150 instances of Iris flowers across 3 species, with 4 morphological measurements.

In [ ]:
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = df['target'].map({0: iris.target_names[0], 1: iris.target_names[1], 2: iris.target_names[2]})

print(f"Dataset shape: {df.shape}")
df.head()

### Pairwise Feature Distribution

In [ ]:
sns.pairplot(df.drop('target', axis=1), hue='species', palette='Set2')
plt.suptitle('Pairwise Feature Distributions by Iris Species', y=1.02)
plt.show()

## 3. Train-Test Split

We split the data into 80% training and 20% holdout test sets, preserving class balance via stratification.

In [ ]:
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 4. The Impact of Feature Scaling

To demonstrate why scaling is vital, let us deliberately introduce an unscaled feature with a wide dynamic range (e.g., multiplying petal length by 10,000).

In [ ]:
# Synthetically introduce scale disparity
X_train_skewed = X_train.copy()
X_test_skewed = X_test.copy()
X_train_skewed[:, 2] = X_train_skewed[:, 2] * 10000.0
X_test_skewed[:, 2] = X_test_skewed[:, 2] * 10000.0

# 1. Unscaled KNN on skewed features
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train_skewed, y_train)
acc_unscaled = accuracy_score(y_test, knn_unscaled.predict(X_test_skewed))

# 2. Properly Scaled KNN using StandardScaler
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
pipeline.fit(X_train_skewed, y_train)
acc_scaled = accuracy_score(y_test, pipeline.predict(X_test_skewed))

print(f"Accuracy without scaling (dominant feature): {acc_unscaled * 100:.2f}%")
print(f"Accuracy with StandardScaler pipeline:       {acc_scaled * 100:.2f}%")

## 5. Finding the Optimal K (Elbow / Error Rate Curve)

We evaluate test error rates across values of $K$ ranging from 1 to 40 using 10-fold cross-validation.

In [ ]:
error_rates = []
k_values = list(range(1, 40))

for k in k_values:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    # 10-Fold Cross Validation
    scores = cross_val_score(pipe, X_train, y_train, cv=10, scoring='accuracy')
    error_rates.append(1.0 - np.mean(scores))

# Plot the Error Rate vs K Value
plt.figure(figsize=(10, 6))
plt.plot(k_values, error_rates, color='blue', linestyle='dashed', marker='o',
         markerfacecolor='red', markersize=8)
plt.title('Error Rate vs. K Value (10-Fold CV)', fontsize=14)
plt.xlabel('K Value', fontsize=12)
plt.ylabel('Mean Cross-Validation Error Rate', fontsize=12)
plt.xticks(np.arange(1, 41, 2))
plt.show()

## 6. Systematic Hyperparameter Tuning with GridSearchCV

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'knn__n_neighbors': list(range(1, 31, 2)),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score:   {grid_search.best_score_:.4f}")

## 7. Model Evaluation on Holdout Test Set

In [ ]:
best_knn = grid_search.best_estimator_
y_pred = best_knn.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Holdout Confusion Matrix')
plt.show()